# Lesson 4: 交叉验证与模型选择

## 学习目标
1. 理解为什么需要交叉验证（一次划分不够可靠）
2. 掌握 K-Fold、Stratified K-Fold 交叉验证
3. 学会用交叉验证做超参数调优 (GridSearchCV)
4. 学习曲线（Learning Curve）诊断模型

---

## 为什么需要交叉验证？

单次 train_test_split 的问题:
- 结果受随机种子影响，换一个种子分数可能差很多
- 小数据集时，测试集太小不够可靠
- 无法充分利用数据

交叉验证的核心思想: **让每条数据都当过测试集，取平均分数**

## Part 1: 单次划分 vs 交叉验证

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

iris = load_iris()
X, y = iris.data, iris.target

# 演示: 不同随机种子，accuracy差别有多大
dt = DecisionTreeClassifier(max_depth=3)

scores = []
for seed in range(50):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    dt.fit(X_tr, y_tr)
    scores.append(dt.score(X_te, y_te))

print(f"50次不同划分的accuracy:")
print(f"  最小: {min(scores):.4f}")
print(f"  最大: {max(scores):.4f}")
print(f"  均值: {np.mean(scores):.4f}")
print(f"  标准差: {np.std(scores):.4f}")
print(f"\n波动范围达到 {(max(scores)-min(scores))*100:.1f}%! 单次划分不可靠")

plt.figure(figsize=(10, 3))
plt.plot(scores, 'bo-', markersize=4)
plt.axhline(y=np.mean(scores), color='r', linestyle='--', label=f'Mean={np.mean(scores):.4f}')
plt.xlabel('Random Seed')
plt.ylabel('Accuracy')
plt.title('Accuracy Varies with Different Random Splits')
plt.legend()
plt.savefig('04_random_variance.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: K-Fold 交叉验证

K-Fold 把数据分成 K 份，轮流用 1 份做测试、K-1 份做训练:
```
Fold 1: [Test] [Train] [Train] [Train] [Train]
Fold 2: [Train] [Test] [Train] [Train] [Train]
Fold 3: [Train] [Train] [Test] [Train] [Train]
Fold 4: [Train] [Train] [Train] [Test] [Train]
Fold 5: [Train] [Train] [Train] [Train] [Test]
```

In [ ]:
# 方法1: 最简单的 cross_val_score（一行搞定）
dt = DecisionTreeClassifier(max_depth=3, random_state=42)

cv_scores = cross_val_score(dt, X, y, cv=5, scoring='accuracy')
print("=== 5-Fold 交叉验证 ===")
print(f"每折分数: {cv_scores}")
print(f"平均分数: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"\n这比单次划分可靠多了!")

In [ ]:
# 方法2: 手动控制 KFold（可以看到每折的细节）
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("=== 手动 KFold ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    dt.fit(X_tr, y_tr)
    score = dt.score(X_te, y_te)
    print(f"Fold {fold+1}: 训练集{len(X_tr)}条, 测试集{len(X_te)}条, Accuracy={score:.4f}")

In [ ]:
# StratifiedKFold: 分类问题必备！保持各折中的类别比例
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Stratified KFold (分层) ===")
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    y_te = y[test_idx]
    print(f"Fold {fold+1}: 测试集类别分布 = {np.bincount(y_te)}")

print("\n每折的类别比例基本一致！")
print("\n建议: 分类问题优先用 StratifiedKFold")
print("      cross_val_score 对分类问题默认就是 StratifiedKFold")

## Part 3: GridSearchCV — 超参数调优

用网格搜索 + 交叉验证自动找最佳超参数。

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

# 先留出测试集（这是最终评估用的，交叉验证不碰它）
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

# 定义要搜索的参数范围
param_grid = {
    'max_depth': [2, 3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV: 对每组参数做5折交叉验证
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,                # 5折交叉验证
    scoring='f1',        # 用F1作为评估指标
    n_jobs=-1,           # 并行计算
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f"总共搜索了 {len(grid_search.cv_results_['mean_test_score'])} 种参数组合")
print(f"\n最佳参数: {grid_search.best_params_}")
print(f"最佳交叉验证F1: {grid_search.best_score_:.4f}")

# 用最佳模型在测试集上评估
test_score = grid_search.score(X_test, y_test)
print(f"\n测试集F1 (最终评估): {test_score:.4f}")

In [ ]:
# 查看所有参数组合的结果（取top 10）
import pandas as pd

cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ['param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf',
        'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
top10 = cv_results[cols].sort_values('rank_test_score').head(10)
print("=== Top 10 参数组合 ===")
print(top10.to_string(index=False))

## Part 4: 学习曲线 — 需要更多数据还是更好的模型？

学习曲线展示了随着训练数据量增加，训练和验证分数的变化:
- **两条线都低**: 欠拟合，需要更复杂的模型
- **训练线高、验证线低**: 过拟合，需要更多数据或简化模型
- **两条线都高且接近**: 模型合适

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_lc = [
    ('欠拟合 (depth=1)', DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('合适 (depth=3)', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('过拟合 (depth=None)', DecisionTreeClassifier(max_depth=None, random_state=42)),
]

for ax, (title, model) in zip(axes, models_lc):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_bc, y_bc, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='accuracy', n_jobs=-1
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training')
    ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(0.8, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("观察:")
print("- 欠拟合: 两条线都低，增加数据也没用，需要更复杂的模型")
print("- 合适: 两条线都高且逐渐接近")
print("- 过拟合: 训练分数高，验证分数低，有gap → 需要更多数据或正则化")

## Part 5: 多模型对比实战

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 用 Pipeline 把预处理和模型串起来（避免数据泄露）
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=10000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', random_state=42))
    ])
}

print("=== 5-Fold 交叉验证对比 ===")
results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1')
    results[name] = scores
    print(f"{name:>25}: F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

# 箱线图对比
plt.figure(figsize=(10, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel('F1 Score')
plt.title('Model Comparison (5-Fold CV)')
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
plt.savefig('04_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 6: 练习

### 练习: 用 GridSearchCV 调优 KNN
1. 在乳腺癌数据集上，用 GridSearchCV 搜索 KNN 的最佳 `n_neighbors` (试 1-20)
2. 记得用 Pipeline 加入 StandardScaler
3. 打印最佳参数和交叉验证分数
4. 在测试集上评估最终效果

In [ ]:
# === 你的代码 ===
from sklearn.pipeline import Pipeline

# 1. 构建 Pipeline

# 2. 定义参数网格 (注意Pipeline中参数名格式: 'model__n_neighbors')

# 3. GridSearchCV

# 4. 打印结果


---
## 小结

| 技术 | 用途 | 常用参数 |
|------|------|----------|
| cross_val_score | 快速评估模型 | cv=5 |
| StratifiedKFold | 分类问题交叉验证 | 保持类别比例 |
| GridSearchCV | 暴力搜索最佳超参数 | param_grid + cv |
| learning_curve | 诊断过拟合/欠拟合 | 看gap大小 |
| Pipeline | 避免数据泄露 | 把预处理和模型串起来 |

**下一课: 综合实战项目 — 完整的模型评估流程**